# Single Question RAG Demo - AYM Bireysel Ba?vuru S?resi

This notebook asks one fixed question using the final official RAG stack. It retrieves context, prints the exact LLM prompt, and generates one answer. No Gradio UI is used.


In [ ]:
!python -m pip install -q "huggingface_hub==0.34.4" "transformers==4.51.3" "sentence-transformers>=3.0.0" accelerate bitsandbytes peft faiss-cpu rank-bm25 pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import gc
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {DRIVE_ROOT}')
os.chdir(DRIVE_ROOT)
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OFFICIAL_INDEX = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b'
BASE_RERANKER = 'Qwen/Qwen3-Reranker-8B'
BASE_LLM = 'Qwen/Qwen3-32B'
QUESTION = 'Anayasa Mahkemesine bireysel ba\u015fvuru hakk\u0131 ka\u00e7 g\u00fcn i\u00e7inde kullan\u0131lmal\u0131d\u0131r?'

print('Working directory:', Path.cwd())
print('Device:', device)
print('Official index:', OFFICIAL_INDEX)
print('Index manifest exists:', (OFFICIAL_INDEX / 'index_manifest.json').exists())
print('Question:', QUESTION)


## Prompt Template

The project uses `build_rag_prompt()` from `src/prompting.py`. The prompt instructs the model to answer only from the retrieved legal context, avoid fabricated laws/articles/sources, and use this answer format:

1. K?sa cevap
2. ?artlar / a??klama
3. Dayanak maddeler
4. Not / s?n?rl?l?k


In [ ]:
from src.retrieval import RetrievalEngine
from src.reranking import CrossEncoderReranker
from src.prompting import build_rag_prompt

candidate_k = 30
top_k_context = 10
reranker_batch_size = 4
max_context_chars = 9000

engine = RetrievalEngine(index_root=OFFICIAL_INDEX, device=device)
candidates = engine.dense_search(QUESTION, top_k=candidate_k)

# Important: free the 8B embedding model before loading the 8B reranker.
del engine
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

reranker = CrossEncoderReranker(model_name=BASE_RERANKER, device=device)
retrieved = reranker.rerank(
    query=QUESTION,
    candidates=candidates,
    text_field='retrieval_text',
    top_k=top_k_context,
    batch_size=reranker_batch_size,
)

del reranker
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print('Retrieved sources:')
for i, item in enumerate(retrieved, start=1):
    print(f"[{i}] {item.get('citation_label') or item.get('article_key')} | article_key={item.get('article_key')} | score={item.get('score')}")


In [ ]:
prompt = build_rag_prompt(QUESTION, retrieved, max_context_chars=max_context_chars)
print(prompt)


In [ ]:
from src.generation import load_llm, generate_text

tokenizer, model = load_llm(
    model_name=BASE_LLM,
    device=device,
    load_in_4bit=True,
)
answer = generate_text(
    tokenizer=tokenizer,
    model=model,
    prompt=prompt,
    max_new_tokens=384,
    temperature=0.0,
    top_p=1.0,
    input_max_length=8192,
)
print(answer)
